# Session 1: why you pay back more than you borrowed
**A. Real-world problem.** A friend lends you 10000 units today. A bank does the same, but
it charges *interest*: a price for using its money. How much do you owe after 3 years, and
why does "interest on interest" make the amount grow faster and faster?

**Plan.** (1) simple interest, (2) compound interest, (3) compare them over time,
(4) ask how long it takes for a debt to double.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: simple interest
Interest is charged only on the original amount `P` (the principal), at a yearly rate `r`
(0.10 means 10 percent a year). After `t` years you owe  A = P (1 + r t).
Example: P = 10000, r = 0.10, t = 3  ->  interest 3000, amount 13000.

In [ ]:
def simple_amount(P, r, t):
    """Amount owed after t years with simple interest."""
    return P * (1 + r * t)

## Step 2: compound interest
Now each year's interest is added to the debt, so next year you pay interest on it too:
    after 1 year: P (1 + r),  after 2 years: P (1 + r)^2,  after t years: P (1 + r)^t.
Same example: 10000 * 1.1 * 1.1 * 1.1 = 13310. That is 310 more than simple interest.

In [ ]:
def compound_amount(P, r, t):
    """Amount owed after t whole years with interest added once a year."""
    return P * (1 + r) ** t

## Step 3: how long until the debt doubles?
Simple interest adds the same slice every year, so doubling takes 1 / r years.
Compound interest grows by a *factor*, so we need (1 + r)^t = 2, i.e. t = ln 2 / ln(1 + r).

In [ ]:
def doubling_time_simple(r):
    return 1.0 / r


def doubling_time_compound(r):
    return float(np.log(2.0) / np.log(1.0 + r))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    P, r = 10000.0, 0.10
    for t in (1, 2, 3):
        print(f"year {t}: simple {simple_amount(P, r, t):.0f}, compound {compound_amount(P, r, t):.0f}")
    print(f"doubling time at 10 percent: simple {doubling_time_simple(r):.2f} y, compound {doubling_time_compound(r):.2f} y")
    t = np.linspace(0, 20, 201)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(t, simple_amount(P, r, t), label="simple interest")
    ax.plot(t, compound_amount(P, r, t), label="compound interest")
    ax.axhline(2 * P, color="gray", ls=":")
    ax.set_xlabel("time (years)")
    ax.set_ylabel("amount owed (units)")
    ax.set_title("Borrow 10000 at 10 percent a year")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "simple_vs_compound.png", dpi=120)
    yrs = np.arange(0, 4)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.bar(yrs - 0.2, simple_amount(P, r, yrs) - P, 0.4, label="simple: interest so far")
    ax2.bar(yrs + 0.2, compound_amount(P, r, yrs) - P, 0.4, label="compound: interest so far")
    ax2.set_xlabel("time (years)")
    ax2.set_ylabel("interest owed so far (units)")
    ax2.set_title("The first three years")
    ax2.set_xticks(yrs)
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "first_three_years.png", dpi=120)

## Try it
Do the coding tasks, the README questions (F), then the checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` change the rate to 5 percent and the time to 10 years. Predict the simple and compound amounts before running the code.
2. **Stretch.** Write `extra_from_compounding(P, r, t)` that returns compound minus simple and print it for t from 1 to 5.
3. **Explain.** Explain in your own words, with one example, why a doubling time at simple interest is 1 / r years but at compound interest is shorter.

## Assessment
- **A. Conceptual.** Why does compound interest grow faster than simple interest, even though both use the same rate? When are they equal?
- **B. Numerical.** You borrow 5000 units at 8 percent a year for 2 years. Find the amount owed with simple and with compound interest. Check that your answers are sensible.
- **C. Graph and simulation interpretation.** In the graph for 10000 units at 10 percent, what are the amounts owed after 20 years, and what does the widening gap between the curves tell you?
- **D. Design and optimisation challenge.** Design: a friend offers 10000 units for 5 years at 12 percent simple interest. A bank offers the same loan at 10 percent compound. Which is cheaper over 5 years? Would your answer change at 20 years?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: simple interest on 10000 at 10 percent for 3 years gives 13000",
                   abs(L.simple_amount(10000, 0.10, 3) - 13000) < 1e-9))
    checks.append(("worked example: compound interest gives 13310, which is 310 more",
                   abs(L.compound_amount(10000, 0.10, 3) - 13310) < 1e-6
                   and abs(L.compound_amount(10000, 0.10, 3) - L.simple_amount(10000, 0.10, 3) - 310) < 1e-6))
    checks.append(("hand calculation: 10000 * 1.1 * 1.1 * 1.1 = 13310",
                   abs(10000 * 1.1 * 1.1 * 1.1 - L.compound_amount(10000, 0.10, 3)) < 1e-6))
    checks.append(("zero years or zero rate: nothing is added",
                   L.simple_amount(500, 0.1, 0) == 500 and L.compound_amount(500, 0.0, 7) == 500))
    checks.append(("in year 1 both methods agree (both 11000)",
                   abs(L.simple_amount(10000, 0.10, 1) - L.compound_amount(10000, 0.10, 1)) < 1e-9))
    checks.append(("after 20 years compound 67275 versus simple 30000",
                   round(L.compound_amount(10000, 0.10, 20)) == 67275 and round(L.simple_amount(10000, 0.10, 20)) == 30000))
    checks.append(("doubling time at 10 percent: simple 10 years, compound 7.27 years",
                   abs(L.doubling_time_simple(0.10) - 10.0) < 1e-9 and abs(L.doubling_time_compound(0.10) - 7.2725) < 1e-3))
    checks.append(("compound never falls below simple for t >= 1",
                   all(L.compound_amount(1000, 0.07, t) >= L.simple_amount(1000, 0.07, t) for t in range(1, 40))))
    checks.append(("exercise numbers: 5000 at 8 percent for 2 years is 5800 simple and 5832 compound",
                   abs(L.simple_amount(5000, 0.08, 2) - 5800) < 1e-9 and abs(L.compound_amount(5000, 0.08, 2) - 5832) < 1e-6))
    checks.append(("design numbers: 10000 for 5 years is 16000 at 12 percent simple and 16105.10 at 10 percent compound",
                   abs(L.simple_amount(10000, 0.12, 5) - 16000) < 1e-9 and abs(L.compound_amount(10000, 0.10, 5) - 16105.10) < 1e-6))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
